# CRAFT — End-to-end parameterization walkthrough

Full pipeline using trimethyllysine (KM3) as a worked example. For algorithm details and background see `CRAFT_tutorial.ipynb`.

**Pipeline overview:**
```
Phase 1 (local)   craft-run        cap termini, generate Gaussian + RESP inputs
Phase 2a (HPC)    g16              B3LYP/6-31G* geometry optimisation
Phase 2b (local)  craft-hf-input   extract optimised geometry, write HF input
Phase 2c (HPC)    g16              HF/6-31G(d) single-point for ESP
Phase 3 (local)   craft-amber      espgen → resp → antechamber → prepgen → parmchk2
```

## 0. Prerequisites

1. CRAFT installed: `pip install .`
2. AmberTools installed and `craft-check` passes.
3. Notebook running from the CRAFT repo root (directory containing `KME3/`).

### Optional: 3D visualization

`pip install py3Dmol` — cells without it print a skip message and continue normally.

In [1]:
import subprocess, sys, shutil
from pathlib import Path

# Verify working directory
assert Path('single_AA/KME3.pdb').exists(), \
    'Run this notebook from the CRAFT repo root (the directory containing KME3/)'

print('Working directory OK')

Working directory OK


In [2]:
# Check that CRAFT and AmberTools are available
result = subprocess.run(['craft-check'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

Python packages
----------------------------------------
  [ok]      numpy
  [ok]      pyyaml
  [ok]      rdkit  (full symmetry-based RESP equivalence)

Gaussian
----------------------------------------
  [MISSING] g16 or g09 -- not found in $PATH

AmberTools
----------------------------------------
  [ok]      espgen -> /Users/xv24323/miniconda3/envs/AmberTools25/bin/espgen
  [ok]      resp -> /Users/xv24323/miniconda3/envs/AmberTools25/bin/resp
  [ok]      antechamber -> /Users/xv24323/miniconda3/envs/AmberTools25/bin/antechamber
  [ok]      prepgen -> /Users/xv24323/miniconda3/envs/AmberTools25/bin/prepgen
  [ok]      parmchk2 -> /Users/xv24323/miniconda3/envs/AmberTools25/bin/parmchk2

  $AMBERHOME = /Users/xv24323/miniconda3/envs/AmberTools25
  [ok]      parm10.dat found

One or more required tools are missing. See above.




In [3]:
# -- Visualization setup -------------------------------------------------------
try:
    import py3Dmol
    HAS_PY3DMOL = True
    print('py3Dmol available — visualization enabled.')
except ImportError:
    HAS_PY3DMOL = False
    print('py3Dmol not installed.  Install with: pip install py3Dmol')
    print('Visualization cells will print a skip message and continue.')


def _pdb_with_bfactor(pdb_path, values):
    """Return PDB string with values written into the B-factor column."""
    lines_out = []
    val_iter = iter(values)
    with open(pdb_path) as f:
        for line in f:
            if line.startswith(('ATOM', 'HETATM')):
                v = next(val_iter, 0.0)
                line = f"{line[:60]}{v:6.2f}{line[66:].rstrip()}\n"
            lines_out.append(line)
    return ''.join(lines_out)


def _charge_to_hex(charge, max_val):
    """Map a charge value to a red/white/blue hex colour string."""
    t = max(min(charge / max_val, 1.0), -1.0)
    if t < 0:    # red (negative) → white (zero)
        r, g, b = 255, int(255 * (1 + t)), int(255 * (1 + t))
    else:         # white (zero) → blue (positive)
        r, g, b = int(255 * (1 - t)), int(255 * (1 - t)), 255
    return f'#{r:02x}{g:02x}{b:02x}'


def show_pdb(pdb_path, charges=None, label_charges=False, width=620, height=440,
             sphere_radius=0.3, background='0xffffff'):
    """Display a PDB file with py3Dmol."""
    if not HAS_PY3DMOL:
        print('(visualization skipped — install py3Dmol with: pip install py3Dmol)')
        return
    if charges is not None:
        pdb_str = _pdb_with_bfactor(pdb_path, charges)
    else:
        with open(pdb_path) as f:
            pdb_str = f.read()
    view = py3Dmol.view(width=width, height=height)
    view.addModel(pdb_str, 'pdb')
    view.setBackgroundColor(background)
    if charges is not None:
        lim = max(abs(c) for c in charges) or 1.0
        cs  = {'prop': 'b', 'gradient': 'rwb', 'min': -lim, 'max': lim}
        view.setStyle({}, {'stick': {'colorscheme': cs, 'radius': 0.08},
                           'sphere': {'radius': sphere_radius, 'colorscheme': cs}})
        if label_charges:
            from craft.cap import parse_pdb as _parse_pdb
            for atom, chg in zip(_parse_pdb(pdb_path), charges):
                view.addLabel(f"{atom['name']} {chg:+.3f}",
                              {'position': {'x': float(atom['x']),
                                            'y': float(atom['y']),
                                            'z': float(atom['z'])},
                               'fontSize': 10, 'fontColor': 'white',
                               'backgroundColor': 'black', 'backgroundOpacity': 0.6,
                               'showBackground': True})
    else:
        view.setStyle({}, {'stick': {'radius': 0.08}, 'sphere': {'radius': sphere_radius}})
    view.zoomTo()
    return view.show()


def show_xyz(atoms_xyz, charges=None, label_charges=False, atom_names=None,
             width=620, height=440, sphere_radius=0.3, background='0xffffff'):
    """
    Display a list of (elem, x, y, z) tuples as returned by parse_opt_log.

    Always loads as XYZ format so hydrogen atoms are never silently dropped
    (py3Dmol filters H atoms when reading PDB format, mirroring the convention
    that X-ray structures omit them).  Charge colouring is applied per-atom
    via index selectors rather than through the B-factor column.

    charges       : list of floats aligned with atoms_xyz — colours by charge.
    label_charges : annotate each atom with atom name (or element) + charge value
                    (requires charges).
    atom_names    : list of atom name strings aligned with atoms_xyz (e.g. the
                    'name' field from parse_pdb).  When provided, labels show the
                    atom name (CA, SG, …) instead of the element symbol.
    """
    if not HAS_PY3DMOL:
        print('(visualization skipped — install py3Dmol with: pip install py3Dmol)')
        return

    # Build XYZ string — all atoms (including H) are always preserved.
    lines = [str(len(atoms_xyz)), 'geometry']
    for elem, x, y, z in atoms_xyz:
        lines.append(f'{elem}  {x:.6f}  {y:.6f}  {z:.6f}')

    view = py3Dmol.view(width=width, height=height)
    view.addModel('\n'.join(lines), 'xyz')
    view.setBackgroundColor(background)

    if charges is not None:
        lim = max(abs(c) for c in charges) or 1.0
        # Apply a uniform base style first, then override colour per atom.
        view.setStyle({}, {'stick': {'radius': 0.08}, 'sphere': {'radius': sphere_radius}})
        for idx, ((elem, x, y, z), chg) in enumerate(zip(atoms_xyz, charges)):
            color = _charge_to_hex(chg, lim)
            view.setStyle({'index': idx},
                          {'stick': {'color': color, 'radius': 0.08},
                           'sphere': {'color': color, 'radius': sphere_radius}})
        if label_charges:
            for i, ((elem, x, y, z), chg) in enumerate(zip(atoms_xyz, charges)):
                name = atom_names[i] if atom_names is not None else elem
                view.addLabel(f"{name} {chg:+.3f}",
                              {'position': {'x': float(x), 'y': float(y), 'z': float(z)},
                               'fontSize': 10, 'fontColor': 'black',
                               'backgroundColor': 'white', 'backgroundOpacity': 0.7,
                               'showBackground': True})
    else:
        view.setStyle({}, {'stick': {'radius': 0.08}, 'sphere': {'radius': sphere_radius}})

    view.zoomTo()
    return view.show()

py3Dmol available — visualization enabled.


## 1. Input PDB

Single-residue PDB — a free amino acid or a residue cut from a structure. CRAFT handles all protonation states automatically. KM3 (trimethyllysine) carries a net charge of +2.

In [4]:
# Input PDB — raw uncapped residue
show_pdb('single_AA/KME3.pdb')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 2. Configuration

Set `residue.input_pdb`, `residue.charge`, `residue.position` (`middle` | `cterm` | `nterm`), and `amber.forcefield` (`ff14SB` | `ff19SB`).

In [5]:
with open('single_AA/config_middle.yaml') as f:
    print(f.read())

# CRAFT configuration -- KM3 trimethyllysine, middle position
#
# Run from this directory (KME3/).  Outputs land in KM3/middle/.
#
# -- If CRAFT is installed as a package (pip install .) ----------------------
#
#   Manually (step-by-step):
#     craft-run --config config_middle.yaml             # Phase 1
#     craft-hf-input KM3/middle/KM3_opt.log \
#                    --config config_middle.yaml        # Phase 2b
#     craft-amber    KM3/middle/KM3_hf.log \
#                    --config config_middle.yaml        # Phase 3
#
#   Single SLURM job (full pipeline):
#     craft-slurm --config config_middle.yaml
#     cd KM3/middle && sbatch KM3_craft.sh
#
# -- If running directly from source (no install) ----------------------------
#
#   Manually (step-by-step):
#     python run.py                                     # Phase 1
#     python make_hf_input.py KM3/middle/KM3_opt.log   # Phase 2b
#     python amber_pipeline.py KM3/middle/KM3_hf.log   # Phase 3
#
#   Single SLURM job (full pi

## 3. Phase 1 — Cap termini and generate Gaussian + RESP inputs

`craft-run` caps the residue with ACE/NME, writes the Gaussian optimisation input (`_opt.com`), and writes all RESP input files. Outputs land in `KM3/middle/`.

In [6]:
result = subprocess.run(
    ['craft-run', '--config', 'config_middle.yaml'],
    capture_output=True, text=True, cwd='single_AA'
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    sys.exit(result.returncode)

Step 1 -- Cap termini  [middle]
Input : KME3.pdb  (34 atoms, residue KM3)
  N-terminus : NH3+ -- zwitterionic free amino acid
  C-terminus : COO- -- has O1 (1 extra O), free amino acid C-terminus
  Position   : middle
  Action     : remove 3 H(s) on N, add amide-H, remove O1, add ACE + NME
Output: KM3/middle/KM3_capped.pdb
  ACE  resSeq=1 :  6 atoms
  KM3  resSeq=2 : 31 atoms
  NME  resSeq=3 :  6 atoms
  Total         : 43 atoms

Step 2 -- Geometry-optimisation Gaussian input
Input  : KM3/middle/KM3_capped.pdb  (43 atoms)
Output : KM3/middle/KM3_opt.com
  Charge / mult : 1 / 1
  nproc / mem   : 16 / 512MB

Steps 3-5 -- RESP and prepgen inputs
  resp.in  : KM3/middle/resp.in  (43 atoms, 25 sidechain free)
  resp.qin : KM3/middle/resp.qin  (43 charges, 25 sidechain free)
  KM3.mc          : HEAD=N CA=CA TAIL=C omit 12 cap atoms

All Phase 1 outputs written to: KM3/middle/

Next steps:
  1. Submit KM3/middle/KM3_opt.com to HPC
  2. Copy KM3_opt.log into KM3/middle/, then:
       craft-hf-

In [7]:
# Capped PDB — ACE (left) + KM3 (centre) + NME (right)
show_pdb('single_AA/KM3/middle/KM3_capped.pdb')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [8]:
# Optimised geometry extracted from the Gaussian opt log
from craft.gaussian import parse_opt_log
opt_geom = parse_opt_log('single_AA/KM3/middle/KM3_opt.log')
show_xyz(opt_geom)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 4. Phase 2 — Gaussian on HPC

Submit to your HPC cluster, then run `craft-hf-input` locally:

```bash
# Phase 2a — geometry optimisation (HPC)
g16 < KM3/middle/KM3_opt.com > KM3/middle/KM3_opt.log

# Phase 2b — build HF input (local)
craft-hf-input KM3/middle/KM3_opt.log --config config_middle.yaml

# Phase 2c — HF single-point for ESP (HPC)
g16 < KM3/middle/KM3_hf.com > KM3/middle/KM3_hf.log
```

Or use `craft-slurm --config config_middle.yaml` to generate a single SLURM script.

The cells below use the pre-computed Gaussian logs from the KME3 example.

In [9]:
# Structure coloured by RESP charge — red = negative, white = zero, blue = positive
show_pdb('single_AA/KM3/middle/KM3_capped.pdb', charges=None)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [10]:
result = subprocess.run(
    ['craft-amber', 'KM3/middle/KM3_hf.log', '--config', 'config_middle.yaml'],
    capture_output=True, text=True, cwd='single_AA'
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    sys.exit(result.returncode)

Residue  : KM3  (charge +1)
Position : middle
Log      : KM3/middle/KM3_hf.log
MC file  : /Users/xv24323/Documents/github/CRAFT/single_AA/KM3/middle/KM3.mc


-- espgen ------------------------------------------------------------
  $ espgen -i /Users/xv24323/Documents/github/CRAFT/single_AA/KM3/middle/KM3_hf.log -o esp.dat


-- resp --------------------------------------------------------------
  $ resp -O -i resp.in -o resp.out -p resp.pch -t resp.chg -q resp.qin -e esp.dat

-- antechamber -------------------------------------------------------
  $ antechamber -fi gout -i /Users/xv24323/Documents/github/CRAFT/single_AA/KM3/middle/KM3_hf.log -bk KM3 -fo ac -o KM3.ac -c rc -cf resp.chg -at amber -nc 1
Info: acdoctor mode is on: check and diagnose problems in the input file.
Info: The atom type is set to amber; the options available to the -at flag are
      gaff, gaff2, amber, bcc, abcg2, and sybyl.

-- Check Format for Gaussian Output File --
   Status: pass

-- remap atom names -------

### Outputs

In [11]:
from craft.cap import parse_pdb

# Same view with per-atom charge labels (zoom in to read individual values)
atom_names = [a['name'] for a in parse_pdb('single_AA/KM3/middle/KM3_capped.pdb')]
charges = [float(x) for x in open('single_AA/KM3/middle/resp.chg').read().split()]

show_xyz(opt_geom, label_charges=True, charges=charges, atom_names=atom_names, width=1000, height=650)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 6. Using the outputs in LEaP

```
source leaprc.protein.ff14SB

loadAmberPrep   KM3/middle/KM3.prepin
loadAmberParams KM3/middle/KM3_ff14SB.frcmod

mol = sequence { ACE KM3 NME }
saveAmberParm mol system.prmtop system.inpcrd
quit
```

## 7. All three terminal positions

Run all three positions with the corresponding config files:

In [12]:
positions = [
    ('middle', 'KM3/middle/KM3_hf.log',      'config_middle.yaml'),
    ('cterm',  'KM3/cterm/KM3_cterm_hf.log', 'config_cterm.yaml'),
    ('nterm',  'KM3/nterm/KM3_nterm_hf.log', 'config_nterm.yaml'),
]

for position, hf_log, config in positions:
    sep = '=' * 60
    print(f'\n{sep}')
    print(f'Position: {position}')
    print(sep)
    result = subprocess.run(
        ['craft-amber', hf_log, '--config', config],
        capture_output=True, text=True, cwd='single_AA'
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)


Position: middle
Residue  : KM3  (charge +1)
Position : middle
Log      : KM3/middle/KM3_hf.log
MC file  : /Users/xv24323/Documents/github/CRAFT/single_AA/KM3/middle/KM3.mc


-- espgen ------------------------------------------------------------
  $ espgen -i /Users/xv24323/Documents/github/CRAFT/single_AA/KM3/middle/KM3_hf.log -o esp.dat


-- resp --------------------------------------------------------------
  $ resp -O -i resp.in -o resp.out -p resp.pch -t resp.chg -q resp.qin -e esp.dat

-- antechamber -------------------------------------------------------
  $ antechamber -fi gout -i /Users/xv24323/Documents/github/CRAFT/single_AA/KM3/middle/KM3_hf.log -bk KM3 -fo ac -o KM3.ac -c rc -cf resp.chg -at amber -nc 1
Info: acdoctor mode is on: check and diagnose problems in the input file.
Info: The atom type is set to amber; the options available to the -at flag are
      gaff, gaff2, amber, bcc, abcg2, and sybyl.

-- Check Format for Gaussian Output File --
   Status: pass

-- remap 

In [13]:
# Summary of output files for all three positions
output_files = [
    'single_AA/KM3/middle/KM3.prepin',
    'single_AA/KM3/middle/KM3_ff14SB.frcmod',
    'single_AA/KM3/cterm/KM3_cterm.prepin',
    'single_AA/KM3/cterm/KM3_cterm_ff14SB.frcmod',
    'single_AA/KM3/nterm/KM3_nterm.prepin',
    'single_AA/KM3/nterm/KM3_nterm_ff14SB.frcmod',
]

print(f"{'File':<55} {'Status'}")
print('-' * 62)
for f in output_files:
    status = 'OK' if Path(f).exists() else 'MISSING'
    print(f"{f:<55} {status}")

File                                                    Status
--------------------------------------------------------------
single_AA/KM3/middle/KM3.prepin                         OK
single_AA/KM3/middle/KM3_ff14SB.frcmod                  OK
single_AA/KM3/cterm/KM3_cterm.prepin                    OK
single_AA/KM3/cterm/KM3_cterm_ff14SB.frcmod             OK
single_AA/KM3/nterm/KM3_nterm.prepin                    OK
single_AA/KM3/nterm/KM3_nterm_ff14SB.frcmod             OK


## 8. Bonded residue parameterization

The [tutorial](CRAFT_tutorial.ipynb) this repository provides a detailed explanation of how to parameterize two residues linked by a covalent bond (e.g. disulfide, isopeptide, thioether, etc.), CRAFT parameterizes both simultaneously. The example given is CYA–ASA (a cysteine derivative bonded to an aspartate derivative via a SG–CG bond).